Running on Colab, will download and install neccesary packages.
upload the "newBenchmarks_dataset.xlsx" in the main folder

In [1]:
!pip install tensorboardX
!pip install pysr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 17.7 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/AndreasMadsen/stable-nalu.git

Cloning into 'stable-nalu'...
remote: Enumerating objects: 3003, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 3003 (delta 57), reused 109 (delta 55), pack-reused 2892 (from 1)
Receiving objects: 100% (3003/3003), 14.82 MiB | 14.26 MiB/s, done.
Resolving deltas: 100% (2271/2271), done.


In [3]:
cd stable-nalu

/content/stable-nalu


In [4]:
!python setup.py develop

/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/dist.py:261: UserWarning: Unknown distribution option: 'test_suite'
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/dist.py:261: UserWarning: Unknown distribution option: 'tests_require'
  warnings.warn(msg)
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/pypa/setuptools/issues/917 for details.
        ********************************************************************************

!!
  easy_install.initialize_options(self)
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDepre

In [5]:
import numpy as np
import pandas as pd
import pickle
from datetime import datetime

In [6]:
# import pysr
!pwd ..

/content/stable-nalu


In [7]:
import torch
import torch.nn as nn
from stable_nalu.layer.re_regualized_linear_nac import ReRegualizedLinearNACLayer
from stable_nalu.layer.re_regualized_linear_nalu import ReRegualizedLinearNALULayer

/content/stable-nalu/stable_nalu/functional/gumbel.py:26: SyntaxWarning: invalid escape sequence '\i'
  tau: the temperature used, must be tau \in (0, \infty]. tau < 1
/content/stable-nalu/stable_nalu/layer/nac.py:13: SyntaxWarning: invalid escape sequence '\h'
  Asumming \hat{w} and \hat{m} are sampled from a uniform


In [8]:
SEED=5
rnd=np.random.default_rng(SEED)
torch.manual_seed(SEED)

In [9]:
!ls ../..

bin	 datalab  home	  lib32   media  proc		    root  srv  tools
boot	 dev	  kaggle  lib64   mnt	 python-apt	    run   sys  usr
content  etc	  lib	  libx32  opt	 python-apt.tar.xz  sbin  tmp  var


In [ ]:
pd.ExcelFile(r"../../content/newBenchmarks_dataset.xlsx",).sheet_names

['Sheet1', 'ideal_gas', 'bode', 'Sheet4', 'schechter']

In [ ]:
data= pd.read_excel(r"../../content/newBenchmarks_dataset.xlsx","schechter")
data.shape

(27, 10)

rescale the L and output

In [15]:
data['y'] *= 1e10
data['L'] /= 1e7

In [16]:
data.describe()

,Unnamed: 0,L,y,Unnamed: 3,Unnamed: 4,Unnamed: 5,y (without noise),Unnamed: 7,Unnamed: 8,Unnamed: 9
count,27.000000,27.000000,2.700000e+01,0.0,0.0,27.000000,2.700000e+01,0.0,2.700000e+01,2.700000e+01
mean,13.000000,31.683371,8.598217e+00,NaN,NaN,13.000000,8.389833e-10,NaN,8.389833e-10,4.735491e-24
std,7.937254,62.665479,1.552668e+01,NaN,NaN,7.937254,1.481676e-09,NaN,1.481676e-09,1.296270e-23
min,0.000000,0.106283,3.907915e-08,NaN,NaN,0.000000,4.221978e-18,NaN,4.221978e-18,0.000000e+00
25%,6.500000,0.393318,1.325802e-02,NaN,NaN,6.500000,1.394803e-12,NaN,1.394803e-12,1.716561e-27
50%,13.000000,5.639191,4.520406e-01,NaN,NaN,13.000000,3.812370e-11,NaN,3.812370e-11,4.846761e-26
75%,19.500000,34.239765,1.014247e+01,NaN,NaN,19.500000,1.149393e-09,NaN,1.149393e-09,1.809458e-24
max,26.000000,288.039552,6.028528e+01,NaN,NaN,26.000000,5.584883e-09,NaN,5.584883e-09,5.045802e-23


In [ ]:
unique_train_indexes = list(rnd.choice(data.shape[0], int(data.shape[0]*0.7), replace=False,))

In [18]:
unique_test_indexes = [_ for _ in range(data.shape[0]) if _ not in unique_train_indexes]
len(unique_train_indexes),len(unique_test_indexes)

(18, 9)

In [47]:
data

,Unnamed: 0,L,y,Unnamed: 3,Unnamed: 4,Unnamed: 5,y (without noise),Unnamed: 7,Unnamed: 8,Unnamed: 9
0,0,34.548092,1.251978e-02,NaN,NaN,0,1.362534e-12,NaN,1.362534e-12,1.413639e-26
1,1,5.639191,4.520406e-01,NaN,NaN,1,3.812370e-11,NaN,3.812370e-11,1.227846e-25
2,2,0.110788,6.028528e+01,NaN,NaN,2,5.312551e-09,NaN,5.312551e-09,4.714929e-23
3,3,9.251380,1.744117e-01,NaN,NaN,3,1.821618e-11,NaN,1.821618e-11,4.846761e-26
4,4,0.719453,4.587813e+00,NaN,NaN,4,5.491831e-10,NaN,5.491831e-10,9.305782e-25
5,5,0.403327,1.042629e+01,NaN,NaN,5,1.113841e-09,NaN,1.113841e-09,5.169879e-24
6,6,1.663702,2.548899e+00,NaN,NaN,6,1.933863e-10,NaN,1.933863e-10,0.000000e+00
7,7,288.039552,3.907915e-08,NaN,NaN,7,4.221978e-18,NaN,4.221978e-18,1.525337e-31
8,8,7.385681,2.380065e-01,NaN,NaN,8,2.571858e-11,NaN,2.571858e-11,9.693523e-27
9,9,19.343681,2.580331e-02,NaN,NaN,9,5.020383e-12,NaN,5.020383e-12,2.665719e-26


In [ ]:
schechter_train_X = data.loc[unique_train_indexes,['L']].values
schechter_train_y = data.loc[unique_train_indexes,['y']].values

schechter_test_X = torch.from_numpy(data.loc[unique_test_indexes,['L']].values)
schechter_test_y = torch.from_numpy(data.loc[unique_test_indexes,['y']].values)


In [ ]:
schechter_train_X.shape,schechter_train_y.shape,schechter_test_X.shape,schechter_test_y.shape

((18, 1), (18, 1), torch.Size([9, 1]), torch.Size([9, 1]))

In [ ]:
from torch.utils.data import Dataset, DataLoader
loader = DataLoader(list(zip(schechter_train_X,schechter_train_y)), shuffle=True, batch_size=6)

In [22]:
kwargs_nalu={'eps': 1e-07,'nac_oob': 'clip','regualizer_shape': 'linear','regualizer_z': 0,'mnac_epsilon': 0,'nalu_bias': False,'nalu_two_nac': True,'nalu_two_gate': False,'nalu_mul': 'normal','nalu_gate': 'normal'}
kwargs_nac={'eps': 1e-07,'nac_oob': 'clip','regualizer_shape': 'linear','regualizer_z': 0,'mnac_epsilon': 0,'nalu_bias': False,'nalu_two_nac': False,'nalu_two_gate': False,'nalu_mul': 'normal','nalu_gate': 'normal'}


In [ ]:
class MyModel(nn.Module):
    def __init__(self, in_feats,h_feats, out_feat):
        super(MyModel, self).__init__()
        self.net0=nn.Linear(in_feats , h_feats*2,bias=False)
        self.net1=nn.Linear(h_feats*2 , h_feats,bias=False)
        # self.net1_1=nn.Linear(h_feats*2 , h_feats*2,bias=False)
        self.leaKyrelu=torch.nn.LeakyReLU(0.1)
        self.net_nac=ReRegualizedLinearNACLayer(in_features=h_feats,out_features=h_feats,**kwargs_nac)
        self.net_nalu=ReRegualizedLinearNALULayer(in_features=h_feats,out_features=out_feat,**kwargs_nalu)
    def forward(self,inputs_):
        h=self.net0(inputs_.float())
        h=self.leaKyrelu(h)
        # h=self.net1_1(h)
        # h=self.leaKyrelu(h)
        h=self.net1(h)
        h=self.leaKyrelu(h)
        h=self.net_nac(h)
        h=self.net_nalu(h)
        return h.double()
    def reset_parameters(self):
        torch.nn.init.xavier_uniform(self.net0.weight)
        torch.nn.init.xavier_uniform(self.net1.weight)
        # torch.nn.init.xavier_uniform(self.net1_1.weight)
        self.net_nalu.reset_parameters()
        self.net_nac.reset_parameters()
    def regualizer(self):
        self.net_nalu.regualizer()
        self.net_nac.regualizer()


In [24]:
net=MyModel(1,2,1)
net.reset_parameters()

/tmp/ipykernel_1908/3821075369.py:21: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(self.net0.weight)
/tmp/ipykernel_1908/3821075369.py:22: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(self.net1.weight)


In [25]:
print(f"start {datetime.now()}")

start 2026-06-28 15:55:19.121453


In [26]:
optim = torch.optim.Adam(net.parameters(), lr=0.001, weight_decay=1e-5)#

In [ ]:
best_loss=np.inf
criterion = torch.nn.HuberLoss()
for i in range(10000):
    loss_list=[]
    # net.train()
    for X_t,y_t in loader:
        optim.zero_grad()
        out=net(X_t)
        loss_train_criterion = criterion(y_t,out)
#         loss_train_regualizer =10 * r_w_scale * regualizers['W'] + regualizers['g'] + 0 * regualizers['z'] + 1 * regualizers['W-OOB']
        loss = loss_train_criterion #+ loss_train_regualizer
        loss.retain_grad()
        loss_list.append(loss.item())
        #################
        mea = torch.mean(torch.abs(y_t - out))
        loss.backward()
        optim.step()
    if i%1==0:
        with torch.no_grad():
            test_loss=criterion(schechter_test_y,net(schechter_test_X)).item()
            if test_loss<best_loss:
                best_loss=test_loss
                torch.save(net,f'../../content/models/modelschechter_2inputs_{i}.pt')

                print(f'saved at {i} with error {np.array(loss_list).mean()}, {best_loss}')

saved at 0 with error 6.827979302354641, 11.396131258472845
saved at 1 with error 6.823878889441645, 11.390413764302577
saved at 2 with error 6.818091327567608, 11.385022900016805
saved at 3 with error 6.813384569403676, 11.379484344398763
saved at 4 with error 6.808499115738395, 11.37394534868541
saved at 5 with error 6.804247056280239, 11.368372206393955
saved at 6 with error 6.799729302532682, 11.362785265523327
saved at 7 with error 6.795070232831752, 11.357282277171771
saved at 8 with error 6.79015854940783, 11.351903299008143
saved at 9 with error 6.786189904318817, 11.346328618259552
saved at 10 with error 6.781538784616724, 11.340777489107833
saved at 11 with error 6.776416476729859, 11.335315982593926
saved at 12 with error 6.772584779415748, 11.329585979965167
saved at 13 with error 6.7674241191635645, 11.323961168504635
saved at 14 with error 6.763322153154715, 11.31813406857995
saved at 15 with error 6.758611151437347, 11.312254794837916
saved at 16 with error 6.75387206497

In [28]:
print(f"end {datetime.now()}")

end 2026-06-28 15:56:29.977335


In [29]:
for name, param in net.named_parameters():
    if 'weight' in name:
        print(f"Layer: {name} | Size: {param.size()} | Values : {param.data}")


Layer: net0.weight | Size: torch.Size([4, 1]) | Values : tensor([[-1.0449],
        [ 0.3815],
        [ 0.5988],
        [ 0.1341]])
Layer: net1.weight | Size: torch.Size([2, 4]) | Values : tensor([[ 1.0337,  0.2410, -0.9799,  0.7975],
        [ 1.0208, -1.1483, -1.6355, -0.0303]])


In [30]:
i

9999

In [31]:
loss

tensor(0.1172, dtype=torch.float64, grad_fn=<HuberLossBackward0>)

In [ ]:
net1=torch.load(f'../../content/models/modelschechter_2inputs_6954.pt',weights_only=False)#99842

extrapolation dataset

In [33]:
res_=[]
for L in np.linspace(0.5,30,100)/10:#rnd.uniform(300.,500.,10):
    obs = (0.002/250000000)*np.power((L/25),-1.2)*(np.exp(-1*L/25))*1e10
    model = net1(torch.from_numpy(np.array([[L]]))).detach().numpy()[0][0]
    res_.append([L,model,obs])

In [34]:
res_ = pd.DataFrame(res_,columns=['L','model','Obs'])
res_['diff'] = (res_['Obs'] - res_['model']).abs()

In [35]:
res_['diff'].mean()

np.float64(0.20810849665719225)

In [36]:
res_

,L,model,Obs,diff
0,0.050000,130.057251,138.351988,8.294737
1,0.079798,75.703499,78.857225,3.153726
2,0.109596,52.384567,53.822104,1.437536
3,0.139394,39.600285,40.281330,0.681046
4,0.169192,31.592525,31.887707,0.295182
...,...,...,...,...
95,2.880808,0.906041,0.953139,0.047097
96,2.910606,0.892247,0.940320,0.048072
97,2.940404,0.878763,0.927790,0.049027
98,2.970202,0.865579,0.915540,0.049960


evaluation the best model on training/validation

In [ ]:
data= pd.read_excel(r"../../content/newBenchmarks_dataset.xlsx","schechter")
# data

In [40]:
data['y'] *= 1e10
data['L'] /= 1e7

In [41]:
pd.DataFrame([net1(torch.from_numpy(data['L'].values.reshape(-1,1))).detach().numpy().flatten(),data['y'].values]).T

,0,1
0,0.001618,1.251978e-02
1,0.294155,4.520406e-01
2,51.730289,6.028528e+01
3,0.097684,1.744117e-01
4,5.656230,4.587813e+00
5,11.360560,1.042629e+01
6,1.954265,2.548899e+00
7,0.011591,3.907915e-08
8,0.168110,2.380065e-01
9,0.008096,2.580331e-02


In [42]:
criterion(net1(torch.from_numpy(data['L'].values.reshape(-1,1))),torch.from_numpy(data['y'].values.reshape(-1,1)))

tensor(0.6780, dtype=torch.float64, grad_fn=<HuberLossBackward0>)